In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# DEAP

In [ ]:
DATA_PATH = "../../data/analysis-figures"

In [ ]:
FILE_PATH = "../../data/svm_gridsearch_results.csv"

In [ ]:
df = pd.read_csv(FILE_PATH)

In [ ]:
df

## Mean and balanced accuracies

In [ ]:
df_acc = df[["mean accuracy", "balanced accuracy"]]

In [ ]:
df_acc

In [ ]:
df_acc_mean = df_acc.mean()

In [ ]:
df_acc_mean

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df.index, df["mean accuracy"], marker="o", label="Mean Accuracy")
plt.plot(df.index, df["balanced accuracy"], marker="s", label="Balanced Accuracy")
plt.axhline(df["mean accuracy"].mean(), color="blue", linestyle="--", alpha=0.7,
            label=f"Mean of Mean Accuracy ({df['mean accuracy'].mean():.3f})")
plt.axhline(df["balanced accuracy"].mean(), color="orange", linestyle="--", alpha=0.7,
            label=f"Mean of Balanced Accuracy ({df['balanced accuracy'].mean():.3f})")
# Labels and title
plt.xlabel("Subject")
plt.ylabel("Accuracy")
plt.title("Accuracy per Subject")
plt.legend(
    loc="upper center", 
    bbox_to_anchor=(0.5, -0.15),   # move legend below
    ncol=2,                        # spread items across columns
    frameon=False                  # remove legend box background
)
plt.grid(True)
plt.xticks(df.index)
plt.ylim(0.55, 0.85)  # adjust min and max for more contrast
plt.savefig(f"{DATA_PATH}/deap/accuracies.png")
plt.show()

## F1-Score

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df.index, df["f1-score"], marker="o", color="green", label="F1-Score")
plt.axhline(df["f1-score"].mean(), color="green", linestyle="--", alpha=0.7,
            label=f"Mean of f1-score ({df['f1-score'].mean():.3f})")
# Labels and title
plt.xlabel("Subject")
plt.ylabel("F1-score")
plt.title("F1-score per Subject")
plt.legend(
    loc="upper center", 
    bbox_to_anchor=(0.5, -0.15),   # move legend below
    ncol=2,                        # spread items across columns
    frameon=False                  # remove legend box background
)
plt.grid(True)
plt.xticks(df.index)
plt.savefig(f"{DATA_PATH}/deap/f1-scores.png")
plt.show()

## Hyperparameter tuning

In [ ]:
# Data prep
kernel_counts = df["kernel"].value_counts()
feature_counts = df["k_features"].value_counts().sort_index()
c_counts = df["C"].value_counts().sort_index()
rbf_df = df[df["kernel"] == "rbf"]
gamma_counts = rbf_df["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel Frequency
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count")
axes[0, 1].set_ylabel("Count")

# C Frequency
c_counts.plot(kind="bar", ax=axes[1, 0], title="C")
axes[1, 0].set_ylabel("Count")

# Gamma Frequency (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/deap/hyperparams_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/deap/hyperparams_2x2.pdf")  # for LaTeX
plt.show()

### Better than mean f1-score

In [ ]:
# Compute mean F1-score
mean_f1 = df["f1-score"].mean()

# Filter rows with f1-score above mean
filtered_df = df[df["f1-score"] > mean_f1]

# Data prep
kernel_counts = filtered_df["kernel"].value_counts()
feature_counts = filtered_df["k_features"].value_counts().sort_index()
c_counts = filtered_df["C"].value_counts().sort_index()
rbf_df = filtered_df[filtered_df["kernel"] == "rbf"]
gamma_counts = rbf_df["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel (F1 > Mean)")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count (F1 > Mean)")
axes[0, 1].set_ylabel("Count")

# C
c_counts.plot(kind="bar", ax=axes[1, 0], title="C (F1 > Mean)")
axes[1, 0].set_ylabel("Count")

# Gamma (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (F1 > Mean, RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/deap/hyperparams>mean_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/deap/hyperparams>mean_2x2.pdf")  # for LaTeX
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
for k in df["kernel"].unique():
    subset = df[df["kernel"] == k]
    plt.scatter(subset["k_features"], subset["mean accuracy"], label=k)

plt.xlabel("Number of Features")
plt.ylabel("Mean Accuracy")
plt.title("Accuracy vs. Features (by Kernel)")
plt.legend()
plt.show()

# Boxplot: Accuracy grouped by kernel
df.boxplot(column="mean accuracy", by="kernel", figsize=(6,6))
plt.title("Accuracy Distribution by Kernel")
plt.suptitle("")
plt.show()

# Barplot: Average accuracy for each C
avg_acc_C = df.groupby("C")["mean accuracy"].mean()
avg_acc_C.plot(kind="bar", figsize=(8,6), title="Average Accuracy by C")
plt.ylabel("Mean Accuracy")
plt.show()

# Barplot: Average accuracy for each gamma
avg_acc_gamma = df.groupby("gamma")["mean accuracy"].mean()
avg_acc_gamma.plot(kind="bar", figsize=(8,6), title="Average Accuracy by Gamma")
plt.ylabel("Mean Accuracy")
plt.show()


### Worse than mean f1-score

In [ ]:
# Compute mean F1-score
mean_f1 = df["f1-score"].mean()

# Filter rows with f1-score <= mean
filtered_df = df[df["f1-score"] <= mean_f1]

# Data prep
kernel_counts = filtered_df["kernel"].value_counts()
feature_counts = filtered_df["k_features"].value_counts().sort_index()
c_counts = filtered_df["C"].value_counts().sort_index()
rbf_df = filtered_df[filtered_df["kernel"] == "rbf"]
gamma_counts = rbf_df["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel (F1 ≤ Mean)")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count (F1 ≤ Mean)")
axes[0, 1].set_ylabel("Count")

# C
c_counts.plot(kind="bar", ax=axes[1, 0], title="C (F1 ≤ Mean)")
axes[1, 0].set_ylabel("Count")

# Gamma (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (F1 ≤ Mean, RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/deap/hyperparams_leqmean_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/deap/hyperparams_leqmean_2x2.pdf")  # for LaTeX
plt.show()

In [ ]:
# Mean F1
mean_f1 = df["f1-score"].mean()

# Split datasets
above_mean = df[df["f1-score"] > mean_f1]
below_equal_mean = df[df["f1-score"] <= mean_f1]

def get_grouped(df_high, df_low, col, numeric_sort=False, filter_rbf=False):
    # Optional: filter RBF only
    if filter_rbf:
        df_high = df_high[df_high["kernel"] == "rbf"]
        df_low = df_low[df_low["kernel"] == "rbf"]

    # Cast to int only if numeric_sort=True
    if numeric_sort:
        series_high = df_high[col].astype(int)
        series_low = df_low[col].astype(int)
    else:
        series_high = df_high[col].astype(str)
        series_low = df_low[col].astype(str)

    counts = pd.DataFrame({
        "F1 > mean": series_high.value_counts(),
        "F1 ≤ mean": series_low.value_counts()
    }).fillna(0)

    # Sorting
    if numeric_sort:
        counts = counts.sort_index()
    else:
        counts = counts.sort_index()

    # Normalize to proportions
    return counts.div(counts.sum(axis=0), axis=1)

# Prepare data
kernel_counts   = get_grouped(above_mean, below_equal_mean, "kernel")
feature_counts  = get_grouped(above_mean, below_equal_mean, "k_features", numeric_sort=True)
c_counts        = get_grouped(above_mean, below_equal_mean, "C")
gamma_counts    = get_grouped(above_mean, below_equal_mean, "gamma", filter_rbf=True)

# --- Plot all in one 2x2 subplot ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

kernel_counts.plot(kind="bar", ax=axes[0,0], title="Kernel")
axes[0,0].set_ylabel("Proportion")

feature_counts.plot(kind="bar", ax=axes[0,1], title="Feature Count")
axes[0,1].set_ylabel("Proportion")

c_counts.plot(kind="bar", ax=axes[1,0], title="C")
axes[1,0].set_ylabel("Proportion")

gamma_counts.plot(kind="bar", ax=axes[1,1], title="Gamma (RBF only)")
axes[1,1].set_ylabel("Proportion")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/deap/hyperparams_grouped_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/deap/hyperparams_grouped_2x2.pdf")  # for LaTeX
plt.show()

# Headset Data

In [60]:
PATH = "../../data/svm_grid_summary_headset.csv"

In [61]:
df2 = pd.read_csv(PATH)

In [ ]:
df2

## Accuracies

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df2.index, df2["mean accuracy"], marker="o", label="Mean Accuracy")
plt.axhline(df2["mean accuracy"].mean(), color="blue", linestyle="--", alpha=0.7,
            label=f"Mean of Mean Accuracy ({df2['mean accuracy'].mean():.3f})")
# Labels and title
plt.xlabel("Session")
plt.ylabel("Accuracy")
plt.title("Accuracy per Session")
plt.legend(
    loc="upper center", 
    bbox_to_anchor=(0.5, -0.15),   # move legend below
    ncol=2,                        # spread items across columns
    frameon=False                  # remove legend box background
)
plt.grid(True)
plt.xticks(df2.index)
plt.savefig(f"{DATA_PATH}/headset/accuracies.png")
plt.show()

## f1-score

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df2.index, df2["f1-score"], marker="o", color="green", label="F1-Score")
plt.axhline(df2["f1-score"].mean(), color="green", linestyle="--", alpha=0.7,
            label=f"Mean of f1-score ({df2['f1-score'].mean():.3f})")
# Labels and title
plt.xlabel("Session")
plt.ylabel("F1-score")
plt.title("F1-score per Session")
plt.legend(
    loc="upper center", 
    bbox_to_anchor=(0.5, -0.15),   # move legend below
    ncol=2,                        # spread items across columns
    frameon=False                  # remove legend box background
)
plt.grid(True)
plt.xticks(df2.index)
plt.savefig(f"{DATA_PATH}/headset/f1-scores.png")
plt.show()

## Hyperparameter

In [ ]:
# Data prep
kernel_counts = df2["kernel"].value_counts()
feature_counts = df2["k_features"].value_counts().sort_index()
c_counts = df2["C"].value_counts().sort_index()
rbf_df2 = df2[df2["kernel"] == "rbf"]
gamma_counts = rbf_df2["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel Frequency
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count")
axes[0, 1].set_ylabel("Count")

# C Frequency
c_counts.plot(kind="bar", ax=axes[1, 0], title="C")
axes[1, 0].set_ylabel("Count")

# Gamma Frequency (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/headset/hyperparams_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/headset/hyperparams_2x2.pdf")  # for LaTeX
plt.show()

### Better than mean f1-score

In [ ]:
# Compute mean F1-score
mean_f1 = df2["f1-score"].mean()

# Filter rows with f1-score above mean
filtered_df2 = df2[df2["f1-score"] > mean_f1]

# Data prep
kernel_counts = filtered_df2["kernel"].value_counts()
feature_counts = filtered_df2["k_features"].value_counts().sort_index()
c_counts = filtered_df2["C"].value_counts().sort_index()
rbf_df2 = filtered_df2[filtered_df2["kernel"] == "rbf"]
gamma_counts = rbf_df2["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel (F1 > Mean)")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count (F1 > Mean)")
axes[0, 1].set_ylabel("Count")

# C
c_counts.plot(kind="bar", ax=axes[1, 0], title="C (F1 > Mean)")
axes[1, 0].set_ylabel("Count")

# Gamma (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (F1 > Mean, RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/headset/hyperparams>mean_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/headset/hyperparams>mean_2x2.pdf")  # for LaTeX
plt.show()

### Worse than f1-mean

In [ ]:
# Compute mean F1-score
mean_f1 = df2["f1-score"].mean()

# Filter rows with f1-score <= mean
filtered_df2 = df2[df2["f1-score"] <= mean_f1]

# Data prep
kernel_counts = filtered_df2["kernel"].value_counts()
feature_counts = filtered_df2["k_features"].value_counts().sort_index()
c_counts = filtered_df2["C"].value_counts().sort_index()
rbf_df2 = filtered_df2[filtered_df2["kernel"] == "rbf"]
gamma_counts = rbf_df2["gamma"].astype(str).value_counts()

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Kernel
kernel_counts.plot(kind="bar", ax=axes[0, 0], title="Kernel (F1 ≤ Mean)")
axes[0, 0].set_ylabel("Count")

# Feature Count
feature_counts.plot(kind="bar", ax=axes[0, 1], title="Feature Count (F1 ≤ Mean)")
axes[0, 1].set_ylabel("Count")

# C
c_counts.plot(kind="bar", ax=axes[1, 0], title="C (F1 ≤ Mean)")
axes[1, 0].set_ylabel("Count")

# Gamma (RBF only)
gamma_counts.plot(kind="bar", ax=axes[1, 1], title="Gamma (F1 ≤ Mean, RBF Kernel)")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/headset/hyperparams_leqmean_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/headset/hyperparams_leqmean_2x2.pdf")  # for LaTeX
plt.show()

In [ ]:
# Mean F1
mean_f1 = df2["f1-score"].mean()

# Split datasets
above_mean = df2[df2["f1-score"] > mean_f1]
below_equal_mean = df2[df2["f1-score"] <= mean_f1]

def get_grouped(df2_high, df2_low, col, numeric_sort=False, filter_rbf=False):
    # Optional: filter RBF only
    if filter_rbf:
        df2_high = df2_high[df2_high["kernel"] == "rbf"]
        df2_low = df2_low[df2_low["kernel"] == "rbf"]

    # Cast to int only if numeric_sort=True
    if numeric_sort:
        series_high = df2_high[col].astype(int)
        series_low = df2_low[col].astype(int)
    else:
        series_high = df2_high[col].astype(str)
        series_low = df2_low[col].astype(str)

    counts = pd.DataFrame({
        "F1 > mean": series_high.value_counts(),
        "F1 ≤ mean": series_low.value_counts()
    }).fillna(0)

    # Sorting
    if numeric_sort:
        counts = counts.sort_index()
    else:
        counts = counts.sort_index()

    # Normalize to proportions
    return counts.div(counts.sum(axis=0), axis=1)

# Prepare data
kernel_counts   = get_grouped(above_mean, below_equal_mean, "kernel")
feature_counts  = get_grouped(above_mean, below_equal_mean, "k_features", numeric_sort=True)
c_counts        = get_grouped(above_mean, below_equal_mean, "C")
gamma_counts    = get_grouped(above_mean, below_equal_mean, "gamma", filter_rbf=True)

# --- Plot all in one 2x2 subplot ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

kernel_counts.plot(kind="bar", ax=axes[0,0], title="Kernel")
axes[0,0].set_ylabel("Proportion")

feature_counts.plot(kind="bar", ax=axes[0,1], title="Feature Count")
axes[0,1].set_ylabel("Proportion")

c_counts.plot(kind="bar", ax=axes[1,0], title="C")
axes[1,0].set_ylabel("Proportion")

gamma_counts.plot(kind="bar", ax=axes[1,1], title="Gamma (RBF only)")
axes[1,1].set_ylabel("Proportion")

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/headset/hyperparams_grouped_2x2.png", dpi=300)
plt.savefig(f"{DATA_PATH}/headset/hyperparams_grouped_2x2.pdf")  # for LaTeX
plt.show()